# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1.
2.
3.
4.
5.

1) a / 2 даст [1. 2. 3.], а a // 4 — [0 1 1]: / выполняет обычное деление, // — целочисленное деление с округлением вниз

2) [4. 4. 4.]: axis=1 суммирует элементы каждой строки, а в каждой строке четыре единицы

3) [[0 1 2], [1 2 3], [2 3 4]]: broadcasting складывает вектор-строку и вектор-столбец, образуя матрицу 3 × 3

4) 6.0: квадратные корни чисел 1, 4 и 9 равны 1, 2 и 3; их сумма равна 6

5) 7.5: квадраты чисел от 1 до 4 — 1, 4, 9, 16; среднее равно 7.5

In [18]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [19]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [20]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [28]:
# 1
months = np.array(["2026-09", "2026-10"])
revenue = np.array([amount[clean & (month == m)].sum() for m in months])
change_pct = (revenue[1] - revenue[0]) / revenue[0] * 100
print("Строк всего:", len(oid), "| в чистых данных:", clean.sum())
print("Выручка сентября:", revenue[0])
print("Выручка октября:", revenue[1])
print(f"Изменение: {change_pct:.2f}%")

Строк всего: 5463 | в чистых данных: 4739
Выручка сентября: 35243410.0
Выручка октября: 34419520.0
Изменение: -2.34%


После очистки данных осталось 4 739 оплаченных заказов. Выручка сентября составила 35 243 410, октября 34 419 520. Выручка снизилась на 2,34%. Результат отличается от ДЗ №2, потомучто в ДЗ №4 были удалены дубли и тестовые заказы

Для менеджера: выручка Дала Маркет за октябрь составила 34,42 млн тенге, снизившись на 2.34% по сравнению с сентябрем

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [29]:
#2
months = np.array(["2026-09", "2026-10"])
categories = np.unique(cat)
cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum()
                       for m in months] for c in categories])
print("Категории:", categories)
print("Выручка по категориям за сентябрь, октябрь:\n", cat_table)
total_by_category = cat_table.sum(axis=1)
category_change_pct = (cat_table[:, 1] - cat_table[:, 0]) / cat_table[:, 0] * 100
print("Выручка за два месяца:", total_by_category)
print("Больше всего выручки:", categories[total_by_category.argmax()])
print("Изменение по категориям %:", np.round(category_change_pct, 2))
print("Сильнее всего упала:", categories[category_change_pct.argmin()])

Категории: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
Выручка по категориям за сентябрь, октябрь:
 [[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Выручка за два месяца: [10123520.  2206210.  5289940. 17672830. 34370430.]
Больше всего выручки: Электроника
Изменение по категориям %: [ 2.56  7.1   2.61 -7.21 -2.5 ]
Сильнее всего упала: Одежда


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [31]:
# 3
order_counts = np.array([[(clean & (cat == c) & (month == m)).sum()
                          for m in months] for c in categories])
average_check = cat_table / order_counts
print(order_counts)
print("Средний чек:\n", np.round(average_check, 2))
print("Самый высокий средний чек:", categories[average_check[:, 1].argmax()])

[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
Средний чек:
 [[10499.43 10768.47]
 [ 4020.    3815.75]
 [ 6187.09  6158.6 ]
 [13520.8  13045.6 ]
 [32772.75 33600.2 ]]
Самый высокий средний чек: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [32]:
# 4
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum()
                   for m in months] for c in cities])
share = table / table.sum(axis=0)
print("Города:", cities)
print("Выручка по городам:\n", table)
print("Доли городов:\n", np.round(share, 4))
print("Сумма долей по каждому месяцу:", share.sum(axis=0))
shymkent_idx = np.where(cities == "Шымкент")[0][0]
print("Доля Шымкента в сентябрь и октябрь %:", np.round(share[shymkent_idx] * 100, 2))
print("Изменение доли Шымкента:", round((share[shymkent_idx, 1] - share[shymkent_idx, 0]) * 100, 2))


Города: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
Выручка по городам:
 [[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
Доли городов:
 [[0.0731 0.0738]
 [0.4115 0.4254]
 [0.2563 0.2571]
 [0.0262 0.0282]
 [0.1035 0.098 ]
 [0.1293 0.1175]]
Сумма долей по каждому месяцу: [1. 1.]
Доля Шымкента в сентябрь и октябрь %: [12.93 11.75]
Изменение доли Шымкента: -1.19


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [33]:
# 5
city_change = table[:, 1] - table[:, 0]
total_change = table[:, 1].sum() - table[:, 0].sum()
contribution = city_change / total_change
print("Изменение выручки по городам:", city_change)
print("Общее изменение выручки:", total_change)
print("Вклад городов в общее падение %:", np.round(contribution * 100, 2))
print(f"Вклад Шымкента: {contribution[shymkent_idx] * 100:.2f}%")

Изменение выручки по городам: [ -36540.  141950. -183690.   45870. -276840. -514640.]
Общее изменение выручки: -823890.0
Вклад городов в общее падение %: [  4.44 -17.23  22.3   -5.57  33.6   62.46]
Вклад Шымкента: 62.46%


Для менеджера: в первую очередь стоит проверить причины снижения выручки в Шымкенте,на него приходится около 62.46% общего падения

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [37]:
# 6
# Первоначальный вариант неверен: table имеет форму (число городов, 2), а table.sum(axis=1)
# имеет форму (число городов,). Такой вектор не выравнивается по строкам при broadcasting
# keepdims=True сохраняет форму (число городов, 1), поэтому деление идёт внутри каждой строки
share_inside_city = table / table.sum(axis=1, keepdims=True)
print("Доля сентября и октября внутри каждого города:\n", np.round(share_inside_city, 4))
print("Сумма долей в каждой строке:", share_inside_city.sum(axis=1))

Доля сентября и октября внутри каждого города:
 [[0.5036 0.4964]
 [0.4976 0.5024]
 [0.5051 0.4949]
 [0.4879 0.5121]
 [0.5197 0.4803]
 [0.5299 0.4701]]
Сумма долей в каждой строке: [1. 1. 1. 1. 1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [38]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
outlier_count = (np.abs(z) > 3).sum()
outlier_share = outlier_count / len(z)
print("Количество заказов с |z| > 3:", outlier_count)
print(f"Их доля: {outlier_share:.4%}")
print("Среднее z:", z.mean())
print("Стандартное отклонение z:", z.std())

Количество заказов с |z| > 3: 87
Их доля: 1.8358%
Среднее z: 7.04695264417065e-17
Стандартное отклонение z: 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [40]:
# 8
valid = clean & ~np.isnan(days)
X = np.column_stack((amount[valid], days[valid]))
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("Размер матрицы X:", X.shape)
print("Минимумы исходных признаков:", X.min(axis=0))
print("Максимумы исходных признаков:", X.max(axis=0))
print("Минимумы после масштабирования:", X_scaled.min(axis=0))
print("Максимумы после масштабирования:", X_scaled.max(axis=0))
# Масштабирование приводит признаки к одному диапазону, чтобы признак с большими числами не доминировал над остальными только изза единиц измерения

Размер матрицы X: (4595, 2)
Минимумы исходных признаков: [720.   1.]
Максимумы исходных признаков: [1.3904e+05 8.0000e+00]
Минимумы после масштабирования: [0. 0.]
Максимумы после масштабирования: [1. 1.]
